# Lecture automatique de documents — comparaison des approches sur GPU

Ce notebook installe l'API et compare, sur les mêmes documents, les approches de la note de projet :
- **Vision** : Qwen-VL lit directement l'image et renvoie les champs ;
- **OCR puis modèle** : un OCR (RapidOCR ou PaddleOCR-VL) lit le texte, puis Qwen range les champs.

**Avant de commencer** : menu **Exécution → Modifier le type d'exécution → GPU T4**, puis Enregistrer.
Exécute ensuite les cellules dans l'ordre (bouton ▶ ou Maj+Entrée).

⚠️ Les fichiers envoyés ici sont sur les serveurs de Google : **documents fictifs ou anonymisés uniquement**.
Tout est effacé à la fin de la session.

## 1. Vérifier que la carte graphique est active

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "PAS DE GPU : active-le dans Exécution > Modifier le type d'exécution"

## 2. Envoyer le projet
Choisis le fichier **Projet1_colab.zip**.

In [ ]:
from google.colab import files
import zipfile

envoi = files.upload()
nom_zip = next(iter(envoi))
with zipfile.ZipFile(nom_zip) as z:
    z.extractall("/content/Projet1")
%cd /content/Projet1
!ls

## 3. Installer les dépendances Python (~1 min)

In [ ]:
!pip install -q -r requirements.txt

## 4. Installer et démarrer Ollama (~1 min)

In [ ]:
!apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

## 5. Télécharger le modèle

Le plan prévoit **Qwen-VL 7b** ; le GPU T4 (16 Go) le fait tourner sans problème.
Mets `qwen2.5vl:3b` pour comparer avec ce que tu as sur ton PC.

In [ ]:
MODELE = "qwen2.5vl:7b"   # ou "qwen2.5vl:3b"

!ollama pull {MODELE}

with open(".env", "w") as f:
    f.write(f"MODEL_NAME={MODELE}\n")
    f.write("OLLAMA_HOST=http://localhost:11434\n")
print(open(".env").read())

## 6. (Facultatif) Installer PaddleOCR-VL (~3-5 min)

Nécessaire seulement pour tester le moteur OCR prévu par le plan. Installation lourde
(PaddlePaddle pour GPU). Si cette cellule échoue, la comparaison fonctionne quand même
avec RapidOCR ; voir les instructions à jour sur https://www.paddleocr.ai

In [ ]:
!python --version
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr[doc-parser]"
# Test dans un processus séparé : affiche la vraie erreur sans bloquer ce notebook
!python -c "from paddleocr import PaddleOCRVL; print('PaddleOCR-VL : OK')"

## 7. (Facultatif) Lancer les tests automatiques

In [ ]:
!python -m pytest -q -p no:warnings

## 8. Comparer les approches sur tes documents

Choisis le type de document, puis envoie une ou plusieurs images / PDF **du même type**.
Le premier passage est plus lent (chargement des modèles en mémoire).

In [ ]:
import json, subprocess, time
import ollama
import pandas as pd
from google.colab import files
from api.config import settings
from api.core.extraction import extraire

TYPE = "facture"          # ou "avis_echeance" (liste : dossier api/document_types)

# PaddleOCR-VL seulement si la cellule 6 a affiché « PaddleOCR-VL : OK »
paddle_ok = subprocess.run(["python", "-c", "from paddleocr import PaddleOCRVL"],
                           capture_output=True).returncode == 0
CONFIGS = [("vision", "rapidocr"), ("ocr", "rapidocr")] + ([("ocr", "paddleocr_vl")] if paddle_ok else [])
print("Configurations comparées :", CONFIGS)

# Mise en route : charge le modèle en mémoire AVANT de chronométrer
print("Chargement du modèle…")
ollama.chat(model=settings.model_name, messages=[{"role": "user", "content": "Bonjour"}],
            options={"num_ctx": settings.num_ctx}, keep_alive="30m")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)  # doit indiquer « 100% GPU »

documents = files.upload()
lignes = []
try:
    for nom, contenu in documents.items():
        for approche, moteur in CONFIGS:
            settings.ocr_moteur = moteur
            config = approche if approche == "vision" else f"ocr + {moteur}"
            try:
                r = extraire(contenu, TYPE, approche)
            except Exception as e:
                print(f"{nom} / {config} : ERREUR {type(e).__name__} : {e}")
                continue
            print(f"{nom} / {config} : {r['duree_s']} s, {r['nb_a_verifier']} champ(s) à vérifier, source={r['source']}")
            ligne = {"document": nom, "config": config, "duree_s": r["duree_s"], "a_verifier": r["nb_a_verifier"]}
            ligne.update({champ: c["valeur"] for champ, c in r["champs"].items()})
            lignes.append(ligne)
finally:
    settings.ocr_moteur = "rapidocr"   # remet le réglage par défaut pour les cellules suivantes

pd.set_option("display.max_columns", None)
tableau = pd.DataFrame(lignes)
tableau

## 9. Voir la réponse complète de l'API pour un document (format envoyé à l'interface)

In [ ]:
APPROCHE = "ocr"          # ou "vision"
settings.ocr_moteur = "rapidocr"

documents = files.upload()
for nom, contenu in documents.items():
    print(f"===== {nom}")
    print(json.dumps(extraire(contenu, TYPE, APPROCHE), ensure_ascii=False, indent=2))

## En cas de problème

- **Erreur de connexion à Ollama** : relance la cellule 4.
- **Le modèle est introuvable** : relance la cellule 5.
- Journal d'Ollama : exécute `!tail -30 /content/ollama.log`
- **« PDX has already been initialized »** : menu Exécution → Redémarrer la session, puis relance
  les cellules 2, 4, 5 et 8 (les installations sont conservées).